# Notebook 5: Appendix Figures and Advanced Visualizations

This notebook is dedicated to generating advanced visualizations that are suitable for an appendix in a research paper or for deeper exploratory analysis. It leverages the `OfflineReplayAnalyzer` to load data and the `FigureGenerator` to create plots.

**Visualizations Generated:**
1. **Q4 Performance Boxplot**: Compares the distribution of journey times for each policy.
2. **Tail Risk Bar Chart**: Compares tail-risk metrics (Q90, CVaR) across policies.
3. **Pareto Frontier**: Visualizes the trade-off between efficiency (mean time) and risk (CVaR).
4. **Hysteresis Trade-off**: Shows the relationship between recommendation stability (switch rate) and performance.
5. **Causal Effects Plot (Conceptual)**: A conceptual plot to illustrate how one might visualize heterogeneous treatment effects.

**Instructions:**
- As before, paste the `experiment_id` from Notebook 03 into the designated cell.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.analytics import OfflineReplayAnalyzer, FigureGenerator

# Setup plotting style
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (12, 7)

### 1. Set Experiment ID and Load Data

In [ ]:
# PASTE YOUR EXPERIMENT ID HERE
experiment_id = "<PASTE YOUR EXPERIMENT ID HERE>"

output_dir = Path(project_root) / 'output'

if "<PASTE" in experiment_id:
    print("⚠️ Please replace the placeholder with your actual experiment ID.")
else:
    print(f"Analyzing Experiment ID: {experiment_id}")
    analyzer = OfflineReplayAnalyzer(experiment_id)
    fig_gen = FigureGenerator(output_dir)
    try:
        analyzer.load_data()
        analyzer.prepare_data()
        print("Data loaded successfully.")
    except Exception as e:
        print(f"❌ Failed to load data: {e}")

### 2. Generate Core Performance Plots

These plots are generated by the `FigureGenerator` class and saved to the `output/` directory.

In [ ]:
if analyzer.df_log is not None:
    print("Generating Q4 boxplot...")
    fig_gen.plot_q4_boxplots(analyzer.df_log)
    
    print("Generating Q4 tail risk bar chart...")
    df_q4_metrics = analyzer._analyze_q4_performance()
    fig_gen.plot_q4_tail_risk(df_q4_metrics)
    
    print("Generating E6 CVaR by scenario plot...")
    df_e6_metrics = analyzer._analyze_e6_performance()
    fig_gen.plot_e6_cvar(df_e6_metrics)
    
    print("\n✅ Core plots saved to the 'output' directory.")
else:
    print("Data not loaded, skipping plot generation.")

### 3. Advanced Visualization: Pareto Frontier (Risk vs. Efficiency)

This plot helps visualize the trade-off between minimizing the average journey time (efficiency) and minimizing the worst-case journey time (risk). The ideal policies are in the bottom-left corner.

In [ ]:
if analyzer.df_log is not None:
    df_summary = analyzer._generate_policy_summary()

    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_summary, x='APT_mean', y='CVaR_0.9', hue='policy_group', s=150, style='policy_group', palette='magma')

    for i, row in df_summary.iterrows():
        plt.text(row['APT_mean'] + 0.1, row['CVaR_0.9'], row['policy_group'], fontsize=9)

    plt.title('Pareto Frontier: Efficiency (APT) vs. Risk (CVaR)', fontsize=16)
    plt.xlabel('Average Process Time (minutes)', fontsize=12)
    plt.ylabel('Conditional Value-at-Risk at 90% (minutes)', fontsize=12)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.legend(title='Policy', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Save and show
    fig_gen._save_plot("pareto_frontier.png")
    plt.show()
else:
    print("Data not loaded.")

### 4. Advanced Visualization: Hysteresis Trade-off

Hysteresis (penalizing recommendation changes) improves user experience but might lead to slightly suboptimal choices. This plot shows the relationship between recommendation stability (`switch_rate`) and performance.

In [ ]:
if 'df_summary' in locals() and not df_summary.empty:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=df_summary, x='switch_rate', y='APT_mean', hue='policy_group', s=150)
    
    # Highlight the two policies with and without hysteresis
    cvar_hys = df_summary[df_summary['policy_group'] == 'cvar']
    cvar_nohys = df_summary[df_summary['policy_group'] == 'cvar-nohys']
    
    if not cvar_hys.empty and not cvar_nohys.empty:
        plt.plot([cvar_hys['switch_rate'].iloc[0], cvar_nohys['switch_rate'].iloc[0]], 
                 [cvar_hys['APT_mean'].iloc[0], cvar_nohys['APT_mean'].iloc[0]], 
                 color='red', linestyle='--', marker='o')

    plt.title('Hysteresis Trade-off: Stability vs. Performance', fontsize=16)
    plt.xlabel('Switch Rate (Fraction of recommendations changed)', fontsize=12)
    plt.ylabel('Average Process Time (minutes)', fontsize=12)
    plt.grid(True)
    plt.legend(title='Policy')
    
    fig_gen._save_plot("hysteresis_tradeoff.png")
    plt.show()
else:
    print("Summary data not available.")